In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler


In [3]:
df = pd.read_csv(r"C:\Users\Admin\Downloads\customer_credit_risk_messy_2000.csv")

df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,11460,55.0,Male,East,Secondary,Self-Employed,525473.648696,1.419765e+06,Education,659.898797,2,29,21.732944,2021-04-06,0
1,10505,46.0,Male,West,Secondary,Self-Employed,266584.819330,1.281594e+05,Car,736.196557,2,38,15.948261,2015-01-30,0
2,11283,NaN,Female,East,Post-Graduate,NaN,464375.173403,1.746137e+05,Home,569.847826,1,28,21.025101,2017-03-20,1
3,10313,NaN,Male,East,Graduate,Salaried,715454.845512,7.322187e+03,Education,713.828371,1,28,36.840187,2024-04-05,0
4,10127,51.0,Male,West,Graduate,Salaried,473166.285880,1.361827e+06,Car,684.527971,3,30,18.281480,2020-12-13,0


In [4]:
df.describe()

,customer_id,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,default_flag
count,2000.000000,1840.000000,1.840000e+03,2.000000e+03,1840.000000,2000.000000,2000.000000,2000.000000,2000.00000
mean,11000.500000,43.896739,6.210302e+05,3.187015e+05,652.428807,1.976000,30.083500,41.734077,0.19350
std,577.494589,14.931316,5.164715e+05,3.871532e+05,83.792316,1.373451,5.465845,20.511603,0.39514
min,10001.000000,18.000000,-3.591476e+06,3.559558e+01,428.286467,0.000000,15.000000,-8.581847,0.00000
25%,10500.750000,31.000000,4.080595e+05,8.740884e+04,595.175374,1.000000,26.000000,30.508528,0.00000
50%,11000.500000,44.000000,5.955118e+05,2.140321e+05,649.767853,2.000000,30.000000,40.370678,0.00000
75%,11500.250000,56.000000,7.768680e+05,4.240870e+05,706.151041,3.000000,34.000000,51.110269,0.00000
max,12000.000000,69.000000,6.170289e+06,7.434983e+06,1022.979964,9.000000,51.000000,225.978066,1.00000


In [23]:
df.isnull().sum()

customer_id          0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
loan_amount          0
loan_purpose         0
credit_score         0
repayment_history    0
transaction_count    0
spending_ratio       0
join_date            0
default_flag         0
dtype: int64

# Part C Data cleaning

In [19]:
df["age"].fillna(df["age"].mean(),inplace=True)

In [20]:
df["gender"].fillna(df["gender"].mode()[0],inplace=True)

KNN imputer


In [22]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

scaler = StandardScaler()
df_numeric_scaled = scaler.fit_transform(df[numeric_cols])

knn = KNNImputer(n_neighbors=5)
df_numeric_imputed_scaled = knn.fit_transform(df_numeric_scaled)

df_numeric_imputed = scaler.inverse_transform(df_numeric_imputed_scaled)

df[numeric_cols] = df_numeric_imputed



MICE imputer

In [10]:

numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

scaler = StandardScaler()
df_numeric_scaled = scaler.fit_transform(df[numeric_cols])

mice = IterativeImputer(max_iter=10, random_state=42)
df_numeric_imputed_scaled = mice.fit_transform(df_numeric_scaled)

df_numeric_imputed = scaler.inverse_transform(df_numeric_imputed_scaled)
df[numeric_cols] = df_numeric_imputed

mode_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = mode_imputer.fit_transform(df[categorical_cols])




# Part D OUtlier handling

zscoreeeeee!!!!!!!!!!!!!!!!!!!!!!!!


In [35]:

from scipy.stats import zscore



numeric_cols = df.select_dtypes(include=[np.number]).columns

df_clean = df.dropna(subset=numeric_cols)

z_scores = np.abs(zscore(df_clean[numeric_cols]))

df_no_outliers = df_clean[(z_scores < 3).all(axis=1)]

print(df.shape)
print(df_no_outliers.shape)

df_no_outliers.to_csv("customer_credit_risk_zscore_cleaned.csv", index=False)


(2000, 15)
(1894, 15)


IQR rangee


In [36]:


num = df.select_dtypes(include="number")

Q1 = num.quantile(0.25)
Q3 = num.quantile(0.75)
IQR = Q3 - Q1

df = df[~((num < (Q1 - 1.5*IQR)) | (num > (Q3 + 1.5*IQR))).any(axis=1)]

print(df.shape)


(1395, 15)


In [38]:


num = df.select_dtypes(include="number")

low = num.quantile(0.01)
high = num.quantile(0.99)

df = df[~((num < low) | (num > high)).any(axis=1)]

print(df.shape)


(1222, 15)


In [39]:


num = df.select_dtypes(include="number")

low = num.quantile(0.05)
high = num.quantile(0.95)

df[num.columns] = num.clip(lower=low, upper=high, axis=1)

print(df.shape)


(1222, 15)


# Part E feature eng.

In [ ]:

num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(exclude="number").columns

In [40]:
df["join_date"] = pd.to_datetime(df["join_date"])

df["join_year"] = df["join_date"].dt.year
df["join_month"] = df["join_date"].dt.month
df["join_day"] = df["join_date"].dt.day
df["join_weekday"] = df["join_date"].dt.weekday


In [41]:
edu_order = {
    "High School": 1,
    "Bachelor": 2,
    "Master": 3,
    "PhD": 4
}

df["education_level"] = df["education_level"].map(edu_order)


In [42]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["gender"] = le.fit_transform(df["gender"])


In [43]:
df = pd.get_dummies(df, columns=["region", "loan_purpose"], drop_first=True)


In [44]:
df["income_group"] = pd.cut(df["annual_income"], bins=4,
labels=["Low", "Medium", "High", "Very High"])


In [45]:
df["high_income_flag"] = (df["annual_income"] > 50000).astype(int)


In [46]:
df["income_quantile"] = pd.qcut(df["annual_income"], q=4,
labels=False)


In [47]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
df["income_kmeans_bin"] = kmeans.fit_predict(df[["annual_income"]])


# Part F

In [8]:

from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer


num_cols = df.select_dtypes(include=['int64','float64']).columns

# 1. Standardization (Z-score)
standard_scaler = StandardScaler()
df_standard = df.copy()
df_standard[num_cols] = standard_scaler.fit_transform(df_standard[num_cols])

In [ ]:
# 2️⃣ MinMaxScaler + KNN

scaler_mm = MinMaxScaler()
df_mm_scaled = scaler_mm.fit_transform(df[numeric_cols])

df_mm_imputed_scaled = knn.fit_transform(df_mm_scaled)

df_mm_imputed = scaler_mm.inverse_transform(df_mm_imputed_scaled)

df_minmax = df.copy()
df_minmax[numeric_cols] = df_mm_imputed

In [12]:

# 3️⃣ MaxAbsScaler + KNN

scaler_ma = MaxAbsScaler()
df_ma_scaled = scaler_ma.fit_transform(df[numeric_cols])

df_ma_imputed_scaled = knn.fit_transform(df_ma_scaled)

df_ma_imputed = scaler_ma.inverse_transform(df_ma_imputed_scaled)

df_maxabs = df.copy()
df_maxabs[numeric_cols] = df_ma_imputed

In [13]:
# 4️⃣ RobustScaler + KNN
scaler_rb = RobustScaler()
df_rb_scaled = scaler_rb.fit_transform(df[numeric_cols])

df_rb_imputed_scaled = knn.fit_transform(df_rb_scaled)

df_rb_imputed = scaler_rb.inverse_transform(df_rb_imputed_scaled)

df_robust = df.copy()
df_robust[numeric_cols] = df_rb_imputed

In [14]:
# 5️⃣ Normalizer (No inverse possible)
normalizer = Normalizer()
df_normalized = df.copy()
df_normalized[numeric_cols] = normalizer.fit_transform(df[numeric_cols])

# part G

In [16]:
# function transfrom

from sklearn.preprocessing import FunctionTransformer
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = numeric_cols.drop(['customer_id','default_flag'])

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

log_tf = FunctionTransformer(lambda x: np.log1p(np.abs(x)))
df_log = df.copy()
df_log[numeric_cols] = log_tf.fit_transform(df[numeric_cols])

rec_tf = FunctionTransformer(lambda x: 1/(x+1))
df_reciprocal = df.copy()
df_reciprocal[numeric_cols] = rec_tf.fit_transform(df[numeric_cols])

sqrt_tf = FunctionTransformer(lambda x: np.sqrt(np.abs(x)))
df_sqrt = df.copy()
df_sqrt[numeric_cols] = sqrt_tf.fit_transform(df[numeric_cols])

In [17]:
# power transform
from sklearn.preprocessing import PowerTransformer

pt_yeo = PowerTransformer(method='yeo-johnson')
df_yeo = df.copy()
df_yeo[numeric_cols] = pt_yeo.fit_transform(df[numeric_cols])

df_positive = df[numeric_cols].clip(lower=1)

pt_box = PowerTransformer(method='box-cox')
df_box = df.copy()
df_box[numeric_cols] = pt_box.fit_transform(df_positive)


# part H

Final Report

Missing values in numerical columns were handled using KNN Imputer (k=5), which helped preserve data patterns without removing rows. No observations were dropped, so the dataset size remained 2000 records.

Outliers were identified using statistical methods and handled through scaling and transformations instead of deleting data. This reduced the impact of extreme values.

Categorical variables were encoded using One-Hot Encoding to convert them into numerical format suitable for machine learning models.

Standardization was applied to numerical features to bring them onto a similar scale. Additional transformations were used where required to reduce skewness.

Three new features were created: Debt-to-Income Ratio, Average Monthly Transactions, and Spending-to-Income Ratio. These features improve the understanding of customer financial behavior.

The final dataset has no missing values, properly encoded variables, scaled features, and is ready for machine learning modeling.